<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/21-responsible-and-production-ai/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 21: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import collections
import itertools
import json
import math
import string

import numpy as np
import pandas as pd


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k], tol) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, (list, tuple)) and any(isinstance(i, np.ndarray) for i in x):
        brackets = "()" if isinstance(x, tuple) else "[]"
        return brackets[0] + ",\n           ".join(_short(i, limit) for i in x) + brackets[1]
    else:
        text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_selection_rates(f):
    rng = np.random.default_rng(0)
    pred = rng.integers(0, 2, 300)
    groups = rng.choice(["x", "y", "z"], 300)
    _scenarios("selection_rates", [
        ("the example", lambda: f([1, 0, 1, 1], ["a", "a", "b", "b"]), {"a": 0.5, "b": 1.0}, None),
        ("three groups", lambda: f(pred, groups), {g: float(pred[groups == g].mean()) for g in ["x", "y", "z"]}, None),
        ("a pandas Series", lambda: f(pd.Series([0, 1, 1]), pd.Series(["m", "f", "f"])), {"m": 0.0, "f": 1.0}, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f(pred, groups).values()}), ["float"], None),
        ("different lengths raise", lambda: f([1, 0, 1], ["a", "b"]), Raises(ValueError), None),
    ])


def check_disparate_impact(f):
    pred = np.array([1, 0, 0, 0, 0, 1, 1, 0, 1, 0])
    groups = np.array(["a"] * 5 + ["b"] * 4 + ["c"])
    _scenarios("disparate_impact", [
        ("the example", lambda: f([1, 0, 0, 0, 0, 1, 1, 0, 0, 1], ["a"] * 5 + ["b"] * 4 + ["b"], "b"), {"a": 0.2 / 0.6, "b": 1.0}, None),
        ("three groups", lambda: f(pred, groups, "b"), {"a": 0.2 / 0.75, "b": 1.0, "c": 0.0}, "divide each group's rate by the reference group's"),
        ("a reference with the lower rate", lambda: f(pred, groups, "a"), {"a": 1.0, "b": 0.75 / 0.2, "c": 0.0}, None),
        ("a reference rate of 0 raises", lambda: f(pred, groups, "c"), Raises(ValueError), None),
        ("an unknown reference raises", lambda: f(pred, groups, "d"), Raises(ValueError), None),
    ])


def check_laplace_count(f):
    def reference(count, eps, seed):
        return float(count + np.random.default_rng(seed).laplace(0, 1 / eps))

    def spread(eps):
        rng = np.random.default_rng(5)
        return float(np.std([f(100, eps, rng) for _ in range(4000)]))

    _scenarios("laplace_count", [
        ("epsilon 1", lambda: f(1179, 1.0, np.random.default_rng(0)), reference(1179, 1.0, 0), "rng.laplace(0, 1 / epsilon)"),
        ("epsilon 0.1", lambda: f(1179, 0.1, np.random.default_rng(1)), reference(1179, 0.1, 1), None),
        ("smaller epsilon, more noise", lambda: spread(0.1) > 5 * spread(1.0), True, None),
        ("not rounded", lambda: type(f(10, 0.5, np.random.default_rng(2))).__name__, "float", None),
        ("epsilon 0 raises", lambda: f(10, 0, np.random.default_rng(3)), Raises(ValueError), None),
    ])


def check_k_anonymity(f):
    df = pd.DataFrame({"age": [30, 30, 30, 40, 40, 50, 50, 50, 50],
                       "zip": ["A", "A", "A", "B", "B", "C", "C", "C", "C"],
                       "country": ["x", "x", "x", "y", "y", None, "z", "z", "z"]})
    _scenarios("k_anonymity", [
        ("one column", lambda: f(df, ["age"]), 2, None),
        ("two columns", lambda: f(df, ["age", "zip"]), 2, None),
        ("missing values form their own group", lambda: f(df, ["age", "country"]), 1, "use groupby(..., dropna=False)"),
        ("a unique row", lambda: f(df.iloc[:6], ["age", "zip"]), 1, None),
        ("returns an int", lambda: type(f(df, ["zip"])).__name__, "int", None),
    ])


def check_error_rates_by_group(f):
    rng = np.random.default_rng(1)
    yt, yp = rng.integers(0, 2, 200), rng.integers(0, 2, 200)
    g = rng.choice(["p", "q"], 200)

    def reference():
        out = {}
        for grp in ["p", "q"]:
            t, p = yt[g == grp], yp[g == grp]
            out[grp] = {"tpr": float(p[t == 1].mean()), "fpr": float(p[t == 0].mean())}
        return out

    def no_positives():
        r = f([0, 0, 1], [1, 0, 1], ["a", "a", "b"])["a"]
        return [math.isnan(r["tpr"]), r["fpr"]]

    _scenarios("error_rates_by_group", [
        ("the example", lambda: f([1, 1, 0, 0, 1, 0], [1, 0, 0, 1, 1, 0], ["a", "a", "a", "a", "b", "b"]),
         {"a": {"tpr": 0.5, "fpr": 0.5}, "b": {"tpr": 1.0, "fpr": 0.0}}, None),
        ("random labels", lambda: f(yt, yp, g), reference(), "TPR over real positives, FPR over real negatives"),
        ("a group with no positives", no_positives, [True, 0.5], "return float('nan') when a rate is undefined"),
    ])


def check_threshold_for_tpr(f):
    rng = np.random.default_rng(2)
    scores = rng.random(500)
    labels = (rng.random(500) < scores).astype(int)

    def achieved(target):
        t = f(scores, labels, target)
        tpr = (scores[labels == 1] >= t).mean()
        higher = scores[labels == 1][scores[labels == 1] > t]
        next_up = higher.min() if len(higher) else None
        tpr_next = (scores[labels == 1] >= next_up).mean() if next_up is not None else 0.0
        return [bool(tpr >= target), bool(tpr_next < target)]

    _scenarios("threshold_for_tpr", [
        ("the example, target 0.5", lambda: f([0.9, 0.7, 0.4, 0.2, 0.8, 0.1], [1, 1, 1, 1, 0, 0], 0.5), 0.7, None),
        ("the example, target 0.6", lambda: f([0.9, 0.7, 0.4, 0.2, 0.8, 0.1], [1, 1, 1, 1, 0, 0], 0.6), 0.4, "you need ceil(target * positives)"),
        ("target 1 keeps every positive", lambda: f([0.9, 0.7, 0.4, 0.2], [1, 1, 1, 1], 1.0), 0.2, None),
        ("random data: reaches the target with the highest threshold", lambda: achieved(0.73), [True, True], None),
        ("no positives raise", lambda: f([0.3, 0.4], [0, 0], 0.5), Raises(ValueError), None),
        ("a target of 0 raises", lambda: f([0.3, 0.4], [1, 0], 0.0), Raises(ValueError), None),
    ])


def check_psi(f):
    rng = np.random.default_rng(3)
    base = rng.normal(0, 1, 5000)
    shifted = rng.normal(0.5, 1, 3000)

    def reference(e, a, n_bins=10, eps=1e-4):
        edges = np.unique(np.quantile(e, np.linspace(0, 1, n_bins + 1)))
        edges[0], edges[-1] = -np.inf, np.inf
        pe = np.clip(np.histogram(e, edges)[0] / len(e), eps, None)
        pa = np.clip(np.histogram(a, edges)[0] / len(a), eps, None)
        return float(np.sum((pa - pe) * np.log(pa / pe)))

    ints = rng.integers(0, 4, 2000)
    _scenarios("psi", [
        ("identical samples", lambda: f(base, base), 0.0, None),
        ("a shift of half a standard deviation", lambda: f(base, shifted), reference(base, shifted), None),
        ("values beyond the training range", lambda: f(base, base + 5), reference(base, base + 5), "make the outer edges -inf and +inf"),
        ("5 bins", lambda: f(base, shifted, n_bins=5), reference(base, shifted, 5), None),
        ("many repeated values", lambda: f(ints, np.r_[ints, np.full(500, 3)]), reference(ints, np.r_[ints, np.full(500, 3)]),
         "remove duplicate edges with np.unique"),
        ("returns a float", lambda: type(f(base, shifted)).__name__, "float", None),
    ])


def check_reweighing_weights(f):
    rng = np.random.default_rng(4)
    groups = rng.choice(["a", "b"], 400, p=[0.7, 0.3])
    y = (rng.random(400) < np.where(groups == "a", 0.4, 0.15)).astype(int)

    def weighted_rates():
        w = f(groups, y)
        return [float(np.sum(w * y * (groups == g)) / np.sum(w * (groups == g))) for g in ["a", "b"]]

    _scenarios("reweighing_weights", [
        ("the example", lambda: f(["a"] * 5 + ["b"] * 5, [1, 0, 0, 0, 0, 1, 1, 1, 0, 0])[0], 0.5 * 0.4 / 0.1, None),
        ("weighted positive rates are equal across groups", lambda: weighted_rates(), [float(y.mean())] * 2,
         "P(g) * P(y) / P(g and y)"),
        ("the total weight equals the number of examples", lambda: float(f(groups, y).sum()), 400.0, None),
        ("one weight per example", lambda: f(groups, y).shape, (400,), None),
        ("already independent data gets weights of 1", lambda: f(["a", "a", "b", "b"], [0, 1, 0, 1]), np.ones(4), None),
    ])


def check_validate_request(f):
    schema = {
        "age": {"type": "number", "min": 17, "max": 100},
        "hours": {"type": "number", "min": 1},
        "gain": {"type": "number", "max": 99999, "required": False},
        "sex": {"type": "category", "allowed": ["Female", "Male"]},
        "country": {"type": "category", "allowed": ["US", "Other"], "required": False},
    }
    good = {"age": 40, "hours": 38.5, "sex": "Female"}
    _scenarios("validate_request", [
        ("a valid request", lambda: f(good, schema), [], None),
        ("with optional fields", lambda: f({**good, "gain": 0, "country": "US"}, schema), [], None),
        ("missing required field", lambda: f({"age": 40, "sex": "Male"}, schema), ["missing field: hours"], None),
        ("out of range, both limits", lambda: f({**good, "age": 300}, schema), ["age must be between 17 and 100"], None),
        ("the limits themselves are allowed", lambda: f({**good, "age": 17, "hours": 1}, schema), [], None),
        ("a type error does not stop the other checks", lambda: f({**good, "age": "forty", "hours": 0}, schema),
         ["age must be a number", "hours must be at least 1"], None),
        ("only a minimum", lambda: f({**good, "hours": 0}, schema), ["hours must be at least 1"], None),
        ("only a maximum", lambda: f({**good, "gain": 1e6}, schema), ["gain must be at most 99999"], None),
        ("not a number", lambda: f({**good, "age": "forty"}, schema), ["age must be a number"], None),
        ("a boolean is not a number", lambda: f({**good, "age": True}, schema), ["age must be a number"], "isinstance(True, int) is True"),
        ("unknown category", lambda: f({**good, "sex": "X"}, schema), ["sex has unknown value 'X'"], None),
        ("several problems, in order", lambda: f({"hours": 0, "sex": "X", "hours_per_weak": 40, "extra": 1}, schema),
         ["missing field: age", "hours must be at least 1", "sex has unknown value 'X'", "unexpected field: hours_per_weak", "unexpected field: extra"],
         "schema fields first, in schema order, then unexpected fields"),
    ])


def check_permutation_importance_scores(f):
    rng = np.random.default_rng(5)
    X = rng.normal(size=(300, 4))
    y = 3 * X[:, 0] + X[:, 2] + rng.normal(0, 0.1, 300)
    w = np.array([3.0, 0.0, 1.0, 0.0])

    def predict(A):
        return A @ w

    def neg_mse(t, p):
        return -float(np.mean((t - p) ** 2))

    def reference(seed, repeats=3):
        r = np.random.default_rng(seed)
        base = neg_mse(y, predict(X))
        out = []
        for j in range(4):
            drops = []
            for _ in range(repeats):
                S = X.copy()
                S[:, j] = X[r.permutation(len(X)), j]
                drops.append(base - neg_mse(y, predict(S)))
            out.append(np.mean(drops))
        return np.array(out)

    def unchanged():
        Xc = X.copy()
        f(predict, Xc, y, neg_mse, 2, np.random.default_rng(0))
        return Xc

    _scenarios("permutation_importance_scores", [
        ("matches the reference shuffles", lambda: f(predict, X, y, neg_mse, 3, np.random.default_rng(7)), reference(7),
         "columns outer, repeats inner, one rng.permutation per repeat"),
        ("unused columns score exactly 0", lambda: f(predict, X, y, neg_mse, 2, np.random.default_rng(1))[[1, 3]], np.zeros(2), None),
        ("the most important column first", lambda: int(np.argmax(f(predict, X, y, neg_mse, 2, np.random.default_rng(1)))), 0, None),
        ("X is not changed", unchanged, X, "shuffle a copy"),
    ])


def check_shapley_values(f):
    rng = np.random.default_rng(6)
    background = rng.normal(size=(30, 4))
    x = np.array([1.0, -2.0, 0.5, 3.0])
    w = np.array([2.0, 0.0, -1.0, 0.5])

    def linear(A):
        return A @ w + 1.0

    def interaction(A):
        return A[:, 0] * A[:, 1] + np.sin(A[:, 2])

    def symmetric(A):
        return A[:, 0] + A[:, 1] + 0 * A[:, 2]

    def efficiency():
        phi = f(interaction, x[:3], background[:, :3])
        return float(phi.sum())

    _scenarios("shapley_values", [
        ("a linear model", lambda: f(linear, x, background), w * (x - background.mean(axis=0)), "w_j * (x_j - mean of background column j)"),
        ("an unused feature gets 0", lambda: float(f(linear, x, background)[1]), 0.0, None),
        ("they add up to f(x) minus the average", efficiency,
         float(interaction(x[None, :3])[0] - interaction(background[:, :3]).mean()), None),
        ("identical roles get identical values", lambda: f(symmetric, np.array([1.0, 1.0, 5.0]), np.zeros((5, 3))), np.array([1.0, 1.0, 0.0]), None),
        ("returns one value per feature", lambda: f(interaction, x[:3], background[:, :3]).shape, (3,), None),
    ])


def check_ks_statistic(f):
    from scipy import stats
    rng = np.random.default_rng(7)
    a, b = rng.normal(0, 1, 400), rng.normal(0.3, 1.2, 250)
    ta, tb = rng.integers(0, 5, 300), rng.integers(1, 6, 200)
    _scenarios("ks_statistic", [
        ("identical samples", lambda: f(a, a), 0.0, None),
        ("two normal samples", lambda: f(a, b), float(stats.ks_2samp(a, b).statistic), None),
        ("tied integer data", lambda: f(ta, tb), float(stats.ks_2samp(ta, tb).statistic), None),
        ("no overlap", lambda: f([1, 2, 3], [10, 11]), 1.0, None),
        ("no overlap, other order", lambda: f([10, 11], [1, 2, 3]), 1.0, "evaluate the CDFs at the values of both samples, and take the absolute gap"),
        ("different sizes, one sample a subset", lambda: f(a[:100], a), float(stats.ks_2samp(a[:100], a).statistic), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `selection_rates`

In [2]:
def selection_rates(y_pred, groups) -> dict:
    y_pred, groups = np.asarray(y_pred), np.asarray(groups)
    if len(y_pred) != len(groups):
        raise ValueError("y_pred and groups must have the same length")
    return {g: float(y_pred[groups == g].mean()) for g in pd.unique(groups)}

In [3]:
check_selection_rates(selection_rates)

PASS  all 5 tests passed for selection_rates


Comparing selection rates is the simplest fairness check, and the one most often reported. On its own it can mislead, as the lesson shows: if the groups' base rates differ, a perfectly accurate model has different selection rates too. That's why it's always reported next to error rates.

`pd.unique` keeps the groups in order of first appearance, which makes reports stable from run to run. `.mean()` of 0s and 1s is the share of 1s.

The length check catches a real and common bug: passing the group column of a **different** split (the training groups with test predictions, say). NumPy would raise an error for the boolean mask, but code written with lists or Series can silently misalign, and the fairness report would be meaningless.

### Exercise 2: `disparate_impact`

In [4]:
def disparate_impact(y_pred, groups, reference) -> dict:
    y_pred, groups = np.asarray(y_pred), np.asarray(groups)
    if reference not in set(groups.tolist()):
        raise ValueError(f"reference group {reference!r} not found")
    ref_rate = y_pred[groups == reference].mean()
    if ref_rate == 0:
        raise ValueError("the reference group's selection rate is 0")
    return {g: float(y_pred[groups == g].mean() / ref_rate) for g in pd.unique(groups)}

In [5]:
check_disparate_impact(disparate_impact)

PASS  all 5 tests passed for disparate_impact


A ratio is easier to compare across settings than a difference: a gap of 5 percentage points means much more when the rates are 6% and 1% than when they're 80% and 75%. The ratio shows that: 0.17 against 0.94.

Which group is the reference is a choice, usually the group with the highest selection rate or a historically advantaged group; the ratio is then at most 1 for everyone else. Always say which reference you used.

The four-fifths rule is a rule of thumb from one country's employment guidance, not a universal law and not a definition of fairness: a ratio of 0.81 isn't fair and 0.79 discriminatory. It's a trigger for a closer look, at error rates, at the reasons, and at whether the difference is justified for the use.

### Exercise 3: `laplace_count`

In [6]:
def laplace_count(true_count: int, epsilon: float, rng) -> float:
    if epsilon <= 0:
        raise ValueError("epsilon must be positive")
    return float(true_count + rng.laplace(0, 1 / epsilon))

In [7]:
check_laplace_count(laplace_count)

PASS  all 5 tests passed for laplace_count


The scale `1 / epsilon` comes from the **sensitivity** of a count: adding or removing one person changes it by at most 1. The Laplace mechanism adds noise with scale `sensitivity / epsilon`, which makes the probability of any output change by at most a factor of `e ** epsilon` when one person is added or removed. That's the guarantee: whatever an attacker knows, the released number tells them almost nothing about whether any single person is in the data.

Smaller `epsilon` means stronger privacy and more noise. Each released statistic **spends** some of the privacy budget: answering the same question twice with fresh noise and averaging would cut the noise, which is exactly why the budgets of repeated queries add up.

Not rounding is deliberate here: rounding afterwards is fine (it's "post-processing", which can't weaken the guarantee), but the checks compare the exact value. Real systems also clip results to sensible ranges, for instance never releasing a negative count, which is also post-processing.

### Exercise 4: `k_anonymity`

In [8]:
def k_anonymity(df: pd.DataFrame, quasi_identifiers) -> int:
    return int(df.groupby(list(quasi_identifiers), dropna=False).size().min())

In [9]:
check_k_anonymity(k_anonymity)

PASS  all 5 tests passed for k_anonymity


`groupby(...).size()` counts the rows in each combination of values, and the smallest count is the `k`. A table with `k = 1` contains at least one person who is unique on those columns, and anyone who knows those facts about them can find their row.

`dropna=False` matters more than it looks: by default, `groupby` silently **drops** rows with a missing value in any grouping column. Those rows would vanish from the count, and a person who is unique precisely because their country is missing would be missed. One check has exactly that case.

**Quasi-identifiers** are columns that aren't identifying on their own but are in combination: age, sex, postcode, date of birth. A well-known study by Latanya Sweeney estimated that most Americans could be uniquely identified by just their postcode, birth date and sex. Choosing which columns count as quasi-identifiers is the hard part: it depends on what an attacker might know.

## Medium

### Exercise 5: `error_rates_by_group`

In [10]:
def error_rates_by_group(y_true, y_pred, groups) -> dict:
    y_true, y_pred, groups = np.asarray(y_true), np.asarray(y_pred), np.asarray(groups)
    result = {}
    for g in pd.unique(groups):
        t, p = y_true[groups == g], y_pred[groups == g]
        tpr = float(p[t == 1].mean()) if (t == 1).any() else float("nan")
        fpr = float(p[t == 0].mean()) if (t == 0).any() else float("nan")
        result[g] = {"tpr": tpr, "fpr": fpr}
    return result

In [11]:
check_error_rates_by_group(error_rates_by_group)

PASS  all 3 tests passed for error_rates_by_group


TPR (recall) answers "of the people who deserve the positive outcome, how many get it?", and FPR answers "of the people who don't, how many get it anyway?". Equal opportunity asks for equal TPR; equalised odds asks for equal TPR **and** FPR.

Returning `nan` for a group with no positives is honest: the rate is undefined, not 0. Returning 0 would report that the model "misses every positive" in a group that has none, which is false, and a fairness dashboard built on that would raise false alarms or hide real ones. With small groups this happens surprisingly often, especially after splitting by two attributes at once.

The mean of an empty array is also `nan` in NumPy, but it comes with a warning; checking first makes the intention clear and keeps the output clean.

### Exercise 6: `threshold_for_tpr`

In [12]:
def threshold_for_tpr(scores, y_true, target: float) -> float:
    scores, y_true = np.asarray(scores, dtype=float), np.asarray(y_true)
    if not 0 < target <= 1:
        raise ValueError("target must be in (0, 1]")
    positives = np.sort(scores[y_true == 1])[::-1]
    if len(positives) == 0:
        raise ValueError("no positive examples")
    needed = int(math.ceil(target * len(positives)))
    return float(positives[needed - 1])

In [13]:
check_threshold_for_tpr(threshold_for_tpr)

PASS  all 6 tests passed for threshold_for_tpr


Sorting the positives' scores from high to low, the threshold that catches exactly the top `n` of them is the `n`-th score. To reach a TPR of `target`, you need `ceil(target * number of positives)` of them. `ceil` rather than `round` or `int`: with 4 positives and a target of 0.6, you need 2.4 positives, which means 3, since 2 would give only 0.5.

The highest such threshold is the one that flags the fewest negatives, so it keeps the false positive rate as low as possible while meeting the target.

Ties need no special handling: if several positives share the threshold score, all of them pass, and the TPR can only end up a little **above** the target, which still satisfies "at least".

In practice, choose thresholds on validation data, never on the test set, and check the resulting rates on fresh data: a threshold chosen to hit exactly 0.65 on one sample will land near, not on, 0.65 on another.

### Exercise 7: `psi`

In [14]:
def psi(expected, actual, n_bins: int = 10, epsilon: float = 1e-4) -> float:
    expected, actual = np.asarray(expected, dtype=float), np.asarray(actual, dtype=float)
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected)
    a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, epsilon, None), np.clip(a, epsilon, None)
    return float(np.sum((a - e) * np.log(a / e)))

In [15]:
check_psi(psi)

PASS  all 6 tests passed for psi


Quantile bins hold equal shares of the training data (10% each with 10 bins), so every part of the training distribution is watched equally closely. The PSI then adds up, bin by bin, how much the production share differs from the training share, weighted by the log of their ratio. It's a symmetric version of the Kullback-Leibler divergence from Module 9.

Opening the outer edges to infinity matters: without it, production values beyond the training range (a new maximum, say) would fall outside every bin, simply vanish from the counts, and hide exactly the drift you're looking for.

The `epsilon` floor stops an empty bin from producing `log(0)`, but it also caps how large one empty bin can make the PSI, so the exact value of `epsilon` affects the result a little. Report it.

`np.unique` handles columns with many repeated values, but as the lesson showed, a column that's mostly one value (capital gains, mostly 0) collapses into very few bins and the PSI becomes blind to changes in the rest. For such columns, choose the edges yourself.

### Exercise 8: `reweighing_weights`

In [16]:
def reweighing_weights(groups, y) -> np.ndarray:
    groups, y = np.asarray(groups), np.asarray(y)
    n = len(y)
    weights = np.empty(n, dtype=float)
    for g in pd.unique(groups):
        for label in pd.unique(y):
            mask = (groups == g) & (y == label)
            if mask.any():
                weights[mask] = (np.mean(groups == g) * np.mean(y == label)) / (mask.sum() / n)
    return weights

In [17]:
check_reweighing_weights(reweighing_weights)

PASS  all 5 tests passed for reweighing_weights


The numerator is how common the combination **would** be if group and label were independent; the denominator is how common it actually is. Combinations that are rarer than independence predicts (positives in a disadvantaged group, say) get weights above 1, and over-represented ones get weights below 1. Training with these weights (`model.fit(X, y, sample_weight=w)`) asks the model to learn as if the label didn't depend on the group.

The check confirms the defining property directly: in the weighted data, the positive rate is the same in every group, and the total weight still equals the number of examples.

Reweighing changes the training data, not the model, so it works with any learner that accepts sample weights. Whether it's appropriate is a separate question: it pushes the model towards demographic parity, which (as the lesson discusses) is the right goal for some uses and the wrong one for others, and it needs the group labels at training time.

### Exercise 9: `validate_request`

In [18]:
def validate_request(payload: dict, schema: dict) -> list:
    errors = []
    for name, rule in schema.items():
        if name not in payload:
            if rule.get("required", True):
                errors.append(f"missing field: {name}")
            continue
        value = payload[name]
        if rule["type"] == "number":
            if isinstance(value, bool) or not isinstance(value, (int, float)):
                errors.append(f"{name} must be a number")
                continue
            low, high = rule.get("min"), rule.get("max")
            if (low is not None and value < low) or (high is not None and value > high):
                if low is not None and high is not None:
                    errors.append(f"{name} must be between {low} and {high}")
                elif low is not None:
                    errors.append(f"{name} must be at least {low}")
                else:
                    errors.append(f"{name} must be at most {high}")
        elif rule["type"] == "category":
            if value not in rule["allowed"]:
                errors.append(f"{name} has unknown value {value!r}")
    for name in payload:
        if name not in schema:
            errors.append(f"unexpected field: {name}")
    return errors

In [19]:
check_validate_request(validate_request)

PASS  all 12 tests passed for validate_request


Returning **all** the problems at once, rather than stopping at the first, is kinder to whoever calls the service: they can fix everything in one go. Pydantic, used in the lesson, works the same way.

The boolean rule looks pedantic but catches a real Python quirk: `True` is an instance of `int`, so `isinstance(True, int)` is `True`, and without the explicit check a request with `"age": true` would be accepted as age 1.

Rejecting unexpected fields catches misspellings: a client sending `"hours_per_weak"` would otherwise have the real field reported missing (if required) or silently ignored (if optional), and the model would quietly use a default.

Input validation is the cheapest safety measure in production. A model given a value outside anything it saw in training (an age of 300, a category it has never met) still returns a confident-looking number. Validation turns that silent failure into a clear error, and the error counts are also a useful monitoring signal: a sudden rise usually means something upstream has changed.

## Hard

### Exercise 10: `permutation_importance_scores`

In [20]:
def permutation_importance_scores(predict, X: np.ndarray, y, metric, n_repeats: int, rng) -> np.ndarray:
    X = np.asarray(X)
    baseline = metric(y, predict(X))
    importances = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        drops = []
        for _ in range(n_repeats):
            shuffled = X.copy()
            shuffled[:, j] = X[rng.permutation(len(X)), j]
            drops.append(baseline - metric(y, predict(shuffled)))
        importances[j] = np.mean(drops)
    return importances

In [21]:
check_permutation_importance_scores(permutation_importance_scores)

PASS  all 4 tests passed for permutation_importance_scores


Shuffling a column keeps its distribution (the same values, in a different order) but breaks its link with everything else in the row, including the target. If the model's score drops, the model was using that link. If nothing changes, the model ignored the column, or another column carried the same information.

`X.copy()` before shuffling is essential. Shuffling in place would leave column `j` scrambled for every later column, so later importances would be measured on an increasingly broken dataset, and the caller's data would be destroyed too. One check verifies `X` is unchanged afterwards.

`X[rng.permutation(n), j]` builds the shuffled column from the **original**, so each repeat is an independent shuffle.

Two caveats from the lesson apply. Correlated columns share importance: shuffling one leaves its partner to stand in, so both can look unimportant. And importance measured on the **training** data rewards whatever the model memorised, including noise; always measure on held-out data, as Module 14 showed.

### Exercise 11: `shapley_values`

In [22]:
def shapley_values(f, x: np.ndarray, background: np.ndarray) -> np.ndarray:
    x, background = np.asarray(x, dtype=float), np.asarray(background, dtype=float)
    d = len(x)
    cache = {}

    def v(subset):
        if subset not in cache:
            data = background.copy()
            data[:, list(subset)] = x[list(subset)]
            cache[subset] = float(np.mean(f(data)))
        return cache[subset]

    phi = np.zeros(d)
    for j in range(d):
        others = [i for i in range(d) if i != j]
        for size in range(d):
            weight = math.factorial(size) * math.factorial(d - size - 1) / math.factorial(d)
            for subset in itertools.combinations(others, size):
                phi[j] += weight * (v(tuple(sorted(subset + (j,)))) - v(subset))
    return phi

In [23]:
check_shapley_values(shapley_values)

PASS  all 5 tests passed for shapley_values


The weight `|S|! (d - |S| - 1)! / d!` is the fraction of all `d!` orderings of the features in which exactly the features of `S` come before `j`. So the Shapley value is `j`'s contribution averaged over every order in which features could be revealed, which is what makes it fair in the game-theory sense: it's the only way of sharing out the prediction that is symmetric, gives nothing to features that change nothing, and adds up exactly.

Caching `v(S)` matters: each subset is used by many features, and computing it means evaluating the model on the whole background. With the cache, there are `2 ** d` model evaluations; without it, about `d` times as many. Using sorted tuples as keys makes `{1, 3}` and `{3, 1}` the same entry.

The checks use cases with known answers. For a **linear** model `f(x) = w . x + b`, the Shapley value of feature `j` is exactly `w_j * (x_j - mean of background column j)`. A feature the model ignores gets exactly 0. Two features that play identical roles get identical values.

`2 ** d` grows fast: 20 features means a million subsets. The SHAP library's methods (KernelSHAP, TreeSHAP) approximate or exploit model structure to make this practical, but they estimate exactly the quantity computed here.

### Exercise 12: `ks_statistic`

In [24]:
def ks_statistic(a, b) -> float:
    a, b = np.sort(np.asarray(a, dtype=float)), np.sort(np.asarray(b, dtype=float))
    values = np.concatenate([a, b])
    cdf_a = np.searchsorted(a, values, side="right") / len(a)
    cdf_b = np.searchsorted(b, values, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

In [25]:
check_ks_statistic(ks_statistic)

PASS  all 6 tests passed for ks_statistic


`np.searchsorted(a, t, side="right")` on a sorted array gives the number of elements `<= t`, so dividing by the length gives the empirical CDF at `t`. Doing it for every observed value at once takes one vectorised call per sample instead of a loop.

`side="right"` counts elements `<= t`, which is the definition of the CDF. With `side="left"` you would count elements strictly below `t`, the CDF just before `t`. That happens to give the same maximum here, because the value just before one point equals the value just after the previous point, but sticking to the definition keeps the code easy to check. You do need the values of **both** samples as evaluation points: the biggest gap can sit at a value that appears only in `b`.

The KS statistic is between 0 (identical distributions) and 1 (no overlap at all). It's sensitive to differences anywhere in the distribution, a shift, a change of spread or of shape, and it needs no binning, unlike PSI. Its p-value comes from the statistic and the two sample sizes, which is the part to leave to `scipy`. With large samples, as the lesson warns, tiny statistics become "significant", so in monitoring the statistic itself (an effect size) is usually more useful than the p-value.

## What's next

The capstone in `04_mini_project.ipynb` uses these tools on a new problem: predicting who will say yes to a bank's marketing call. You'll train a model without leaking the answer, audit it by age group, explain it, write its model card, serve it with FastAPI, and watch its inputs drift as the economy changes.